# 🥈 Bronze → Silver v2

**Full data-aware rewrite** based on scanning all 735,136 rows across 1480 CSVs.

### What this notebook fixes vs v1
- Parser missed Home University (H) and Other (O) seat type suffixes → they ended up in `candidate_category`
- Splits `candidate_category` into `(base_category, seat_type, seat_marker)` correctly
- Handles all 3 seat pools: State (S), Home University (H), Other than Home Univ (O)
- Cleans all category suffixes: `$`, `#`, `@`, `/DEF1`, `/PH1`
- Adds `seat_pool` column (State / HomeUniv / Other / AllIndia / Minority / TFWS / EWS)

**Output:** `rankrangers_project_data.silver.mhcet_allotments`

## Cell 1 — Config

In [ ]:
# NOTE: source switched from the raw CSV glob to the Bronze Delta table
# (see bronze_delta_ingest.ipynb) so Silver reads a single frozen,
# schema-enforced snapshot instead of re-scanning the CSVs on every run.
# CSV_ROOT kept for reference / rollback:
# CSV_ROOT = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025_csv'
BRONZE_TABLE = 'rankrangers_project_data.bronze.mhcet_allotments_raw'
CATALOG    = 'rankrangers_project_data'
SCHEMA     = 'silver'
TABLE      = 'mhcet_allotments'
FULL_TABLE = f'{CATALOG}.{SCHEMA}.{TABLE}'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
print(f'Source : {BRONZE_TABLE}')
print(f'Target : {FULL_TABLE}')

## Cell 2 — Load Bronze

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType

# _source_file / _ingest_ts / _batch_id already exist on the Bronze Delta
# table (added by bronze_delta_ingest.ipynb) - no need to derive
# _source_file via input_file_name() anymore.
df = spark.table(BRONZE_TABLE)

raw_count = df.count()
print(f'Raw rows loaded: {raw_count:,}')

## Cell 3 — Remove VACANT & Invalid Rows

In [ ]:
# Remove VACANT seats and rows with no valid application ID
df = df.filter(
    (F.col('is_vacant') == False) &
    (F.col('application_id').isNotNull()) &
    (F.col('application_id') != 'VACANT') &
    F.col('application_id').rlike(r'^EN\d{8}$')
)

# Remove rows with no score
df = df.filter(
    F.col('mhtcet_score').isNotNull()
)

after_count = df.count()
print(f'After removing VACANT/invalid: {after_count:,} (removed {raw_count - after_count:,})')

## Cell 4 — Fix seat_type & seat_marker Embedded in candidate_category

The parser missed H (Home University) and O (Other) seat types.
They ended up concatenated into `candidate_category` like:
```
'OPEN ^ GOPENH'   → base=OPEN  marker=^  seat_type=GOPENH
'OBC GOBCH'       → base=OBC   marker=''  seat_type=GOBCH
'SC ^ GSCO'       → base=SC    marker=^   seat_type=GSCO
'OBC ^ MI-MH'     → base=OBC   marker=^   seat_type=MI-MH
'NT 2 (NT-C)/PH1 ^ PWDRNT2S' → base=NT2  seat_type=PWDRNT2S
```

In [ ]:
from pyspark.sql import functions as F

# Regex to extract marker + seat_type from end of candidate_category
# Covers:
#   State:    GOPENS, LOBCS, GSCS, GNT2S ...
#   HomeUniv: GOPENH, LOBCH, GSCH, GNT2H ...
#   Other:    GOPENO, LOBCO, GSCO, GNT2O ...
#   PWD/DEF:  PWDRNT2S, PWDSTS, DEFOPENS, DEFSCS, PWDOPENH, PWDOBCO ...
#   Special:  MI-MH, EWS, TFWS, ORPHAN, AI

MARKER_RE   = r'([~^*@&])\s+'
SEAT_TYPE_RE = (
    r'('                                                   # group 2: seat_type
    r'[GL](?:OPEN|OBC|SC|ST|SEBC|EWS|VJ|NT[123])[HOS]'  # Home/Other/State variants
    r'|[GL](?:OPEN|OBC|SC|ST|SEBC|EWS|VJ|NT[123])S'      # State with S suffix
    r'|(?:PWD|DEF)R?(?:OPEN|OBC|SC|ST|SEBC|NT[123]|VJ)?[HOS]?'# PWD/DEF variants (State/Home/Other)
    r'|MI-MH|MI'                                           # Minority
    r'|EWS|TFWS|ORPHAN|AI'                                 # Special
    r')\s*$'
)

FULL_RE = MARKER_RE + SEAT_TYPE_RE   # optional marker then seat_type at end
BARE_RE = SEAT_TYPE_RE               # seat_type at end with no marker

# For rows where seat_type is null/empty, try to extract from candidate_category
needs_fix = F.col('seat_type').isNull() | (F.col('seat_type') == '')

# Extract marker (group 1) and seat_type (group 2) when marker present
df = df.withColumn(
    '_extracted_marker',
    F.when(needs_fix, F.trim(F.regexp_extract('candidate_category', FULL_RE, 1)))
     .otherwise(None)
).withColumn(
    '_extracted_seat',
    F.when(needs_fix, F.regexp_extract('candidate_category', FULL_RE, 2))
     .otherwise(None)
)

# If FULL_RE (with marker) found nothing, try BARE_RE (no marker)
df = df.withColumn(
    '_extracted_seat',
    F.when(
        needs_fix & (F.col('_extracted_seat') == ''),
        F.regexp_extract('candidate_category', BARE_RE, 1)
    ).otherwise(F.col('_extracted_seat'))
)

# Apply extracted values
df = df.withColumn(
    'seat_type',
    F.when(
        needs_fix & (F.col('_extracted_seat') != ''),
        F.col('_extracted_seat')
    ).otherwise(F.col('seat_type'))
).withColumn(
    'seat_marker',
    F.when(
        needs_fix & (F.col('_extracted_marker') != '') & F.col('_extracted_marker').isNotNull(),
        F.col('_extracted_marker')
    ).otherwise(F.col('seat_marker'))
)

# Strip extracted seat_type (and marker) from candidate_category
df = df.withColumn(
    'candidate_category',
    F.when(
        F.col('_extracted_seat') != '',
        F.trim(F.regexp_replace('candidate_category', MARKER_RE + SEAT_TYPE_RE, ''))
    ).otherwise(
        F.trim(F.regexp_replace('candidate_category', BARE_RE, ''))
    )
)

# Clean up temp columns
df = df.drop('_extracted_marker', '_extracted_seat')

# Replace empty strings with null
df = df.withColumn('seat_type',   F.when(F.col('seat_type')   == '', None).otherwise(F.col('seat_type')))
df = df.withColumn('seat_marker', F.when(F.col('seat_marker') == '', None).otherwise(F.trim(F.col('seat_marker'))))

still_null = df.filter(F.col('seat_type').isNull()).count()
print(f'Rows still missing seat_type: {still_null:,}')
print(f'Rows with seat_type: {df.filter(F.col("seat_type").isNotNull()).count():,}')

## Cell 5 — Clean candidate_category

In [ ]:
# Strip all suffixes: $, #, @, /DEF1, /PH1, (NT-X) labels, extra whitespace
df = df.withColumn(
    'candidate_category',
    F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col('candidate_category'),
                        r'/(?:DEF|PH)\d+', ''   # remove /DEF1, /PH1
                    ),
                    r'\s*\(NT-[ABCD]\)', ''     # remove (NT-B), (NT-C) etc.
                ),
                r'[$#@]+', ''                    # remove $, #, @
            ),
            r'\s+', ' '                          # collapse multiple spaces
        )
    )
)

# Standardise to clean category names
def standardise_category(col):
    return (
        F.when(col.rlike(r'(?i)^OPEN'),              'OPEN')
         .when(col.rlike(r'(?i)^EWS'),               'EWS')
         .when(col.rlike(r'(?i)^SBC'),               'SBC')
         .when(col.rlike(r'(?i)^SEBC'),              'SEBC')
         .when(col.rlike(r'(?i)^OBC'),               'OBC')
         .when(col.rlike(r'(?i)^SC'),                'SC')
         .when(col.rlike(r'(?i)^ST'),                'ST')
         .when(col.rlike(r'(?i)^DT|^VJ'),            'VJ/DT')
         .when(col.rlike(r'(?i)NT\s*1|NT\s*A'),      'NT1')
         .when(col.rlike(r'(?i)NT\s*2|NT\s*B|NT\s*C'),'NT2')
         .when(col.rlike(r'(?i)NT\s*3|NT\s*D'),      'NT3')
         .when(col.rlike(r'(?i)PWD'),                'PWD')
         .when(col.rlike(r'(?i)DEF'),                'DEFENCE')
         .otherwise(col)
    )

df = df.withColumn('clean_category', standardise_category(F.col('candidate_category')))

print('Category distribution after cleaning:')
df.groupBy('clean_category').count().orderBy(F.col('count').desc()).show(30, truncate=False)

## Cell 6 — Derive seat_gender & seat_pool from seat_type

In [ ]:
# seat_gender: G prefix → M (general/male eligible), L prefix → F (ladies only)
df = df.withColumn(
    'seat_gender',
    F.when(F.col('seat_type').startswith('L'),    'F')
     .when(F.col('seat_type').startswith('G'),    'M')
     .when(F.col('seat_type').isin('AI','MI','MI-MH','EWS','TFWS','ORPHAN'), 'ANY')
     .when(F.col('seat_type').rlike(r'^(?:PWD|DEF)'), 'M')  # default PWD/DEF to M
     .otherwise('M')
)

# seat_pool: what quota/pool is this seat from
df = df.withColumn(
    'seat_pool',
    F.when(F.col('seat_type').endswith('S'),          'State')
     .when(F.col('seat_type').endswith('H'),          'HomeUniv')
     .when(F.col('seat_type').endswith('O'),          'Other')
     .when(F.col('seat_type').isin('AI'),             'AllIndia')
     .when(F.col('seat_type').isin('MI','MI-MH'),     'Minority')
     .when(F.col('seat_type') == 'TFWS',              'TFWS')
     .when(F.col('seat_type') == 'EWS',               'EWS')
     .when(F.col('seat_type') == 'ORPHAN',            'Orphan')
     .when(F.col('seat_type').rlike(r'^PWD'),         'PWD')
     .when(F.col('seat_type').rlike(r'^DEF'),         'Defence')
     .otherwise('Other')
)

print('Seat pool distribution:')
df.groupBy('seat_pool').count().orderBy(F.col('count').desc()).show(truncate=False)

print('Seat gender distribution:')
df.groupBy('seat_gender').count().orderBy(F.col('count').desc()).show()

## Cell 7 — Cast Numeric Columns & Add cap_round_num

In [ ]:
# CAP round as integer for sorting
df = df.withColumn(
    'cap_round_num',
    F.when(F.col('cap_round') == 'CAP-I',   1)
     .when(F.col('cap_round') == 'CAP-II',  2)
     .when(F.col('cap_round') == 'CAP-III', 3)
     .when(F.col('cap_round') == 'CAP-IV',  4)
     .otherwise(None).cast(IntegerType())
)

# Numeric casts
for col_name in ['sr_no','merit_no','sanction_intake','cap_seats','ms_seats','minority_seats','ai_seats']:
    df = df.withColumn(col_name, F.col(col_name).cast(IntegerType()))

df = df.withColumn('mhtcet_score', F.col('mhtcet_score').cast(FloatType()))

# Drop rows where score is still null after cast
before = df.count()
df = df.filter(F.col('mhtcet_score').isNotNull())
after  = df.count()
print(f'Dropped {before-after:,} rows with null score after cast')
print(f'Score range: {df.agg(F.min("mhtcet_score"), F.max("mhtcet_score")).collect()[0]}')

# Load timestamp
df = df.withColumn('_load_ts', F.current_timestamp())

## Cell 8 — Data Quality Report

In [ ]:
total = df.count()
print(f'SILVER DATA QUALITY REPORT')
print(f'===========================')
print(f'Total rows: {total:,}')
print()

# Null check on key columns
# NOTE: only check '== ""' for string columns - comparing a numeric column
# (mhtcet_score, cap_round_num) to an empty string errors under ANSI SQL
# mode (CAST_INVALID_INPUT), found while testing locally on Spark 4.x where
# ANSI mode defaults to on. Numeric columns can't hold '' anyway, so
# isNull() alone is sufficient for them.
key_cols = ['mhtcet_score','seat_type','seat_gender','seat_pool',
            'clean_category','institute_name','branch_name','cap_round_num']
string_cols = {name for name, dtype in df.dtypes if dtype == 'string'}
for col in key_cols:
    condition = F.col(col).isNull()
    if col in string_cols:
        condition = condition | (F.col(col) == '')
    nulls = df.filter(condition).count()
    pct   = round(nulls/total*100, 2)
    icon  = '✅' if nulls == 0 else ('⚠️ ' if pct < 1 else '❌')
    print(f'{icon} {col:25s}: {nulls:>8,} nulls ({pct}%)')

print()
print('Rows per CAP round:')
display(df.groupBy('cap_round','cap_round_num').count().orderBy('cap_round_num'))

print('Seat pool breakdown:')
display(df.groupBy('seat_pool','seat_gender').count().orderBy(F.col('count').desc()))

print('Clean category breakdown:')
display(df.groupBy('clean_category').count().orderBy(F.col('count').desc()))

## Cell 9 — Write Silver Delta Table

In [ ]:
df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', True) \
    .saveAsTable(FULL_TABLE)

written = spark.table(FULL_TABLE).count()
print(f'✅ Written to {FULL_TABLE}')
print(f'   Total rows: {written:,}')

## Cell 10 — Quick Verify

In [ ]:
display(spark.sql(f'''
    SELECT cap_round, cap_round_num, seat_pool, seat_gender, clean_category,
           COUNT(*) as rows,
           ROUND(MIN(mhtcet_score),2) as min_score,
           ROUND(MAX(mhtcet_score),2) as max_score
    FROM {FULL_TABLE}
    GROUP BY cap_round, cap_round_num, seat_pool, seat_gender, clean_category
    ORDER BY cap_round_num, seat_pool, clean_category
'''))